## 1. Выявление признаков (x)

Категориальные признаки

Это признаки, которые принимают ограниченное количество значений (город, пол, тарифный план, цвет).

### Метод А: One-Hot Encoding (OHE) / Dummy-кодирование

**Для чего:** Для линейных моделей (Регрессия, Логистическая регрессия) и нейросетей. **Как работает:** Создает новый бинарный столбец (0 или 1) для каждой уникальной категории.

### Метод Б: Ordinal Encoding (Порядковое кодирование)

**Для чего:** Для деревьев решений и случайного леса. Линейные модели с ним работают плохо (они решат, что категория "3" в три раза важнее категории "1"). **Как работает:** Просто заменяет каждую категорию на целое число (0, 1, 2...).

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# Сырые данные: профили клиентов
data = pd.DataFrame({
    'city': ['Москва', 'Казань', 'Москва', 'Сочи'], # Неупорядоченная категория
    'subscription_level': ['Basic', 'Premium', 'Pro', 'Basic'], # Упорядоченная категория
    'income': [100, 150, 120, 80]
})

# 1. One-Hot Encoding (быстрый способ через pandas, удобен для аналитики)
ohe_data = pd.get_dummies(data, columns=['city'], dtype=int)
print("--- После One-Hot Encoding ---")
print(ohe_data[['income', 'city_Казань', 'city_Москва', 'city_Сочи']].head(3))

# 2. Ordinal Encoding (для уровней подписки, где есть логический порядок)
# Явно задаем порядок: Basic (0) -> Pro (1) -> Premium (2)
ordinal = OrdinalEncoder(categories=[['Basic', 'Pro', 'Premium']])
data['sub_encoded'] = ordinal.fit_transform(data[['subscription_level']])

print("\n--- После Ordinal Encoding ---")
print(data[['subscription_level', 'sub_encoded']].head(3))

**Лайфхак для бустинга:** Современным библиотекам вроде `CatBoost` или `LightGBM` вообще не нужно переводить категории в числа вручную. Вы просто передаете им сырые строки и указываете индексы категориальных колонок — они «под капотом» применяют сложные алгоритмы (Target Encoding), которые работают эффективнее базового OHE.

## 2. Текстовые данные

Если категорий тысячи (например, имена, отзывы, описания товаров), OHE создаст матрицу гигантских размеров, состоящую из нулей. Текст нужно векторизовать иначе.

### Метод: TF-IDF (Term Frequency - Inverse Document Frequency)

**Как работает:** Оценивает важность каждого слова. Если слово часто встречается в конкретном отзыве, но редко во всех остальных (например, специфический термин или название бренда), оно получает высокий вес. Частые предлоги («и», «в», «на») получают вес, близкий к нулю.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Сырые данные: отзывы на ресторан
reviews = [
    "Ужасный сервис, еда холодная и невкусная",
    "Отличный ресторан, вкусная еда, придем еще",
    "Еда нормальная, но сервис мог быть и лучше"
]

# Настраиваем векторизатор. 
# max_features=5 - оставим только 5 самых важных слов во всем наборе, чтобы влезло на экран
tfidf = TfidfVectorizer(max_features=5)
X_text = tfidf.fit_transform(reviews)

print("Топ-5 самых важных слов:", tfidf.get_feature_names_out())
print("\nВекторы отзывов (каждое число - вес слова в предложении):\n", X_text.toarray())

Примечание: Для современных нейросетей (Deep Learning) TF-IDF уже почти не используют. Вместо этого применяют Word Embeddings (Word2Vec, трансформеры), которые переводят слова в плотные векторы, сохраняющие смысловую близость (например, расстояние между векторами «король» и «мужчина» будет таким же, как между «королева» и «женщина»).

## 3. Сборка всего вместе: Идеальный пайплайн (Pipeline)

В реальных задачах у вас в одной таблице есть и числа (возраст), и категории (город), и текст (комментарий), а еще могут быть пустые значения (`NaN`).

Делать предобработку руками для каждого столбца — плохая практика (легко ошибиться на новых данных). Профессионалы используют `ColumnTransformer` и `Pipeline` из scikit-learn.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Сырая таблица
df = pd.DataFrame({
    'age': [25, None, 30, 22], # Есть пропуск (None)
    'city': ['MSK', 'SPB', 'MSK', 'KZN'], # Категория
    'purchased': [1, 0, 1, 0] # Целевая переменная
})

X = df[['age', 'city']]
y = df['purchased']

# Шаг 1: Правила для числовых колонок (заполняем пропуски средним -> масштабируем)
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()) # Сжимает числа в диапазон ~[-3, 3], чтобы большие цифры не ломали модель
])

# Шаг 2: Правила для категорий (заполняем пропуски словом 'missing' -> делаем One-Hot)
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Шаг 3: Объединяем правила для разных колонок
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, ['age']),
    ('cat', cat_transformer, ['city'])
])

# Шаг 4: Собираем финальную трубу: Предобработка + Модель
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

# Теперь обучение запускается одной строкой! 
# Данные сами пройдут через все трансформации и попадут в модель.
model_pipeline.fit(X, y)

# Пришел новый сырой клиент с пропусками - пайплайн сам всё обработает
new_client = pd.DataFrame({'age': [28], 'city': ['SPB']})
print("Предсказание для нового клиента:", model_pipeline.predict(new_client))